Seems like tensorzinb is choking on very high expression... Let's see if we can reproduce it in an isolated environment.

In [34]:
from tensorzinb.tensorzinb import TensorZINB
import statsmodels
import numpy as np
import pandas as pd
from formulaic import Formula

In [12]:
def draw_nb(mu, sigma2, size):
    """
    Draw samples from a Negative Binomial distribution using mean and variance.

    Args:
        mu (float): Desired mean.
        sigma2 (float): Desired variance (must be > mu).
        size (int): Number of samples to draw.

    Returns:
        samples (array): Generated samples.
    """
    if sigma2 <= mu:
        raise ValueError("Variance must be greater than the mean (sigma² > μ)")

    # Convert mean & variance to standard Negative Binomial parameters
    r = mu**2 / (sigma2 - mu)  # Number of successes
    p = mu / sigma2  # Probability of success

    samples = np.random.negative_binomial(n=r, p=p, size=size)
    
    return samples

def draw_zinb(mu, sigma2, zin, size):
    """
        Args:
            mu (float): Desired mean.
            sigma2 (float): Desired variance (must be > mu).
            zin (float): 0-1 Desired fraction zero inflation. 
            size (int): Number of samples to draw.
    """

    zin_size=round(size*zin)
    non_zin_size=size-zin_size
    
    zerolist=pd.Series([0]*zin_size)
    draws=draw_nb(mu=mu,sigma2=sigma2,size=non_zin_size)

    return np.concatenate([zerolist,draws])



In [168]:
alpha=list('ABCDEFGHIJKLMNO')



varconst=1.3

def makeset(means,zi,n):
    ret=pd.DataFrame({'mean':means,'zi':zi})
    ret["variance_constant"] = [varconst]*len(ret)
    ret["sigmasquare"]=ret["mean"]+ret["variance_constant"]*ret["mean"]**2
    ret["regressor"]=alpha[0:len(ret)]
    
    def wrapper(row):
        return draw_zinb(mu=row['mean'],sigma2=row['sigmasquare'],zin=row['zi'],size=n)
    
    ret["draw"]=ret.apply(wrapper,axis=1)

    ret=ret.explode(['draw']).reset_index(drop=True)

    ret['draw']=ret['draw'].astype(int)

    return ret


skinny=makeset([1,2,3,6],[0,0,0,0],n=500)

fat_zi=0
fat=makeset([50,100,400,500],[fat_zi]*4,n=500)

In [126]:
fat

,mean,zi,variance_constant,sigmasquare,regressor,draw
0,50,0.5,1.3,3300.0,A,0
1,50,0.5,1.3,3300.0,A,0
2,50,0.5,1.3,3300.0,A,0
3,50,0.5,1.3,3300.0,A,0
4,50,0.5,1.3,3300.0,A,0
...,...,...,...,...,...,...
1995,500,0.5,1.3,325500.0,D,138
1996,500,0.5,1.3,325500.0,D,70
1997,500,0.5,1.3,325500.0,D,69
1998,500,0.5,1.3,325500.0,D,943


In [169]:
def extract_parameter(data):
    y, X=Formula('draw ~ C(regressor)-1').get_model_matrix(data,output='pandas')
    Z=Formula('C(regressor)').get_model_matrix(data,output='pandas')

    
    zinbo = TensorZINB(y["draw"].to_numpy().reshape((-1, 1)), X, exog_infl=Z.to_numpy())#,same_dispersion=True  
    
    fit = zinbo.fit()

    return fit,pd.DataFrame({'regressor': [i[len('C(regressor)['):-1] for i in X.columns.to_list()],
                  'x_mu':fit['weights']['x_mu'].squeeze()
            })

    


In [170]:
fit,mu=extract_parameter(skinny)

In [171]:
np.exp(mu["x_mu"])

0    1.036957
1    1.804591
2    3.192433
3    6.203847
Name: x_mu, dtype: float32

In [156]:
extract_parameter(fat)

({'llf_total': nan,
  'llfs': array([nan]),
  'aic_total': nan,
  'aics': array([nan]),
  'df_model_total': 9,
  'df': 9,
  'weights': {'x_mu': array([[nan],
          [nan],
          [nan],
          [nan]], dtype=float32),
   'x_pi': array([[nan],
          [nan],
          [nan],
          [nan]], dtype=float32),
   'theta': array([[nan]], dtype=float32)},
  'cpu_time': 0.911757230758667,
  'num_sample': 2000,
  'epochs': 50},
   regressor  x_mu
 0         A   NaN
 1         B   NaN
 2         C   NaN
 3         D   NaN)

If its not sparse enough fat fails to converge. 

If it is sparse, we can recap prior failure mode of very high mean values...

Not fixed by same_dispersion, turning off early stop, or changing init method

In [172]:
fat_flat=fat.copy()
fat_flat["draw"]=fat_flat["draw"]/100

In [173]:
fit,mu=extract_parameter(fat_flat)
mu

,regressor,x_mu
0,A,-0.811182
1,B,0.008532
2,C,1.414804
3,D,1.493369


In [174]:
np.exp(mu["x_mu"])*100

0     44.433250
1    100.856819
2    411.568054
3    445.206940
Name: x_mu, dtype: float32

Not quite.

In [151]:
fat_log=fat.copy()

In [153]:
fat_log["draw"]=np.log(fat_log["draw"])

In [154]:
fat_log

,mean,zi,variance_constant,sigmasquare,regressor,draw
0,50,0.5,1.3,3300.0,A,-inf
1,50,0.5,1.3,3300.0,A,-inf
2,50,0.5,1.3,3300.0,A,-inf
3,50,0.5,1.3,3300.0,A,-inf
4,50,0.5,1.3,3300.0,A,-inf
...,...,...,...,...,...,...
1995,500,0.5,1.3,325500.0,D,4.927254
1996,500,0.5,1.3,325500.0,D,4.248495
1997,500,0.5,1.3,325500.0,D,4.234107
1998,500,0.5,1.3,325500.0,D,6.849066
